### <span style = "color:orange">1. Introduction<span>

#### <span style="color:green">1.1. Project purpose</span>

Xây dựng và đánh giá tập hợp các mô hình dự báo chuỗi thời gian để dự đoán nhiệt độ cực đại hàng ngày tại các khu vực đô thị và ven biển Việt Nam. Dự án áp dụng cả thuật toán Machine Learning truyền thống và Deep Learning hiện đại nhằm cải thiện độ chính xác dự báo so với các phương pháp thống kê thông thường.

<b>Nguồn dữ liệu</b>: Bộ dữ liệu ERA5 (ECMWF) với các bản ghi nhiệt độ từ 1990 đến 2024, được xử lý và biến đổi để huấn luyện các mô hình như Random Forest, XGBoost, LSTM, Transformer, TFT, và N-BEATS.

<b>Kết quả</b>: Đánh giá hiệu năng giữa các mô hình qua nhiều kịch bản thực nghiệm và đề xuất hệ thống cảnh báo nhiệt độ sớm ứng dụng thực tế.

#### <span style="color:green">1.2. Data source and description</span>

<h4>Thông tin dữ liệu trong đề tài:</h4>

<ul>
<li><b>Thời gian thu thập:</b> từ năm <b>1990 đến 2024</b></li>
<li><b>Định dạng ban đầu:</b> .grib, sau đó chuyển đổi sang .csv để xử lý</li>
</ul>

<h4>Các biến số chính trong tập dữ liệu:</h4>

<table>
<thead>
<tr>
<th>Tên cột dữ liệu</th>
<th>Ý nghĩa</th>
<th>Đơn vị đo</th>
</tr>
</thead>
<tbody>
<tr><td><code>NAME</code></td><td>Tên tỉnh/thành phố nơi thu thập dữ liệu</td><td>-</td></tr>
<tr><td><code>LATITUDE</code></td><td>Vĩ độ địa lý của điểm đo</td><td>Độ</td></tr>
<tr><td><code>LONGITUDE</code></td><td>Kinh độ địa lý của điểm đo</td><td>Độ</td></tr>
<tr><td><code>YMD</code></td><td>Ngày/tháng/năm đo đạc</td><td>dd/mm/yyyy</td></tr>
<tr><td><code>YEAR</code></td><td>Năm đo đạc</td><td>Năm</td></tr>
<tr><td><code>MONTH</code></td><td>Tháng đo đạc</td><td>Tháng</td></tr>
<tr><td><code>DAY</code></td><td>Ngày đo đạc</td><td>Ngày</td></tr>
<tr><td><code>TEMP_max</code></td><td>Nhiệt độ không khí cực đại trong ngày</td><td>°C</td></tr>
<tr><td><code>TEMP_ave</code></td><td>Nhiệt độ trung bình trong ngày</td><td>°C</td></tr>
<tr><td><code>DEW_ave</code></td><td>Điểm sương trung bình trong ngày</td><td>°C</td></tr>
<tr><td><code>DEW_max</code></td><td>Điểm sương cao nhất trong ngày</td><td>°C</td></tr>
<tr><td><code>RH_ave</code></td><td>Độ ẩm tương đối trung bình trong ngày</td><td>%</td></tr>
<tr><td><code>RH_max</code></td><td>Độ ẩm tương đối cực đại trong ngày</td><td>%</td></tr>
<tr><td><code>AT_ave</code></td><td>Nhiệt độ cảm nhận trung bình trong ngày (Apparent Temp.)</td><td>°C</td></tr>
<tr><td><code>AT_max</code></td><td>Nhiệt độ cảm nhận cao nhất trong ngày</td><td>°C</td></tr>
</tbody>
</table>

<p><b>Biến mục tiêu chính:</b></p>
<ul>
<li><code>TEMP_max</code> — Nhiệt độ không khí cực đại hàng ngày (°C)</li>
</ul>

<p><b>Lưu ý:</b> Dữ liệu gốc của ERA5 có thể chứa giá trị thiếu, giá trị ngoại lai và một số dị bản khí tượng đặc thù. Do đó, quá trình làm sạch dữ liệu, xử lý giá trị thiếu, phát hiện ngoại lệ và chuẩn hóa dữ liệu là các bước bắt buộc trước khi tiến hành huấn luyện và dự báo.</p>

In [1]:
match_type = {
    'NAME'       : 'Categorical',        # Tên tỉnh/thành phố (chuỗi)
    'LATITUDE'   : 'Numerical',          # Vĩ độ (°)
    'LONGITUDE'  : 'Numerical',          # Kinh độ (°)
    'YMD'        : 'Datetime',           # Ngày/tháng/năm (dd/mm/yyyy)
    'YEAR'       : 'Numerical',          # Năm (năm)
    'MONTH'      : 'Numerical',          # Tháng (1-12)
    'DAY'        : 'Numerical',          # Ngày (1-31)

    'TEMP_max'   : 'Numerical',          # Nhiệt độ cực đại trong ngày (°C)
    'TEMP_ave'   : 'Numerical',          # Nhiệt độ trung bình trong ngày (°C)
    'DEW_ave'    : 'Numerical',          # Điểm sương trung bình trong ngày (°C)
    'DEW_max'    : 'Numerical',          # Điểm sương cực đại trong ngày (°C)
    'RH_ave'     : 'Numerical',          # Độ ẩm tương đối trung bình trong ngày (%)
    'RH_max'     : 'Numerical',          # Độ ẩm tương đối cực đại trong ngày (%)
    'AT_ave'     : 'Numerical',          # Nhiệt độ cảm nhận trung bình trong ngày (°C)
    'AT_max'     : 'Numerical'           # Nhiệt độ cảm nhận cực đại trong ngày (°C)
}

#### <span style="color:green">1.3. Goals</span>

<img src="../../image/Ảnh chụp màn hình 2025-06-24 210616.png">

### <span style="color:orange">2. Import Libraries</span>

#### <span style="color:green">2.1. Configuration and display settings</span>

In [2]:
import sys
sys.path.append("../../Drafts/Temp Prediction/")  # đường dẫn đến thư mục chứa src

from src.utilities import(config, 
                          dataset, 
                          features, 
                          plots)

from src.models import(anomaly_models,
                       forecasting_models,
                       model_utils)

from scripts import(evaluate_model,
                    run_forecast,
                    train_model)

#### <span style="color:green">2.2. Required Python packages</span>

In [3]:
import numpy             as np
import pandas            as pd
import matplotlib.pyplot as plt
import seaborn           as sns
import os

from copy import deepcopy
from datetime import datetime

from sklearn.preprocessing import OneHotEncoder  # Encode feature
from sklearn.preprocessing import OrdinalEncoder # Encode feature
from sklearn.preprocessing import MinMaxScaler   # Scale feature
from sklearn.preprocessing import StandardScaler # Scale feature
from sklearn.preprocessing import LabelEncoder   # Encode target
# from scipy.stats import boxcox # Normalized feature

# from sklearn.feature_selection import mutual_info_classif # PCA
from sklearn.model_selection   import train_test_split
from sklearn.model_selection   import RepeatedKFold
from sklearn.model_selection   import GridSearchCV
from sklearn.model_selection   import validation_curve
from sklearn.model_selection   import learning_curve


from sklearn.ensemble     import RandomForestRegressor
from xgboost              import XGBRegressor

# %pip install tensorflow
import tensorflow as tf
from tensorflow                 import keras
from tensorflow.keras           import layers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten, Input
from tensorflow.keras.optimizers import Adam, SGD

# Load the TensorBoard notebook extension
%load_ext tensorboard
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from tensorboard import notebook
from tensorboard.plugins.hparams import api as hp
from torch.utils.tensorboard import SummaryWriter

import joblib

# Classification
# from sklearn.metrics import accuracy_score
# from sklearn.metrics import matthews_corrcoef
# from sklearn.metrics import confusion_matrix
# from sklearn.metrics import roc_auc_score
# from sklearn.metrics import roc_curve
# from sklearn.metrics import classification_report

# Regression
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_squared_log_error
from sklearn.metrics import mean_absolute_percentage_error
# from sklearn.metrics import median_absolute_error
# from sklearn.metrics import max_error
# from sklearn.metrics import PredictionErrorDisplay

from sklearn.inspection import permutation_importance

### <span style="color:orange">3. Data Loading</span>

In [4]:
src = dict({"CaMau" : "../../Drafts/Temp Prediction/data/processed/datasets/CaMau_90.24_cleaned.csv",
            "DH"    : "../../Drafts/Temp Prediction/data/processed/datasets/DH90.24_cleaned.csv",
            "NB"    : "../../Drafts/Temp Prediction/data/processed/datasets/NB90.24_cleaned.csv",
            "QN"    : "../../Drafts/Temp Prediction/data/processed/datasets/QN90.24_cleaned.csv",
            "TH"    : "../../Drafts/Temp Prediction/data/processed/datasets/TH90.24_cleaned.csv",
            "TSN"   : "../../Drafts/Temp Prediction/data/processed/datasets/TSN90.24_cleaned.csv"})

#### <span style="color:green">3.1. Loading the dataset</span>

In [5]:
station_name = "CaMau"

In [6]:
df = pd.read_csv(filepath_or_buffer = src[station_name],
                 parse_dates        = True,
                 index_col          = "time")

#### <span style="color:green">3.2. Displaying first few rows</span>

In [7]:
df.head()

,NAME,LATITUDE,LONGITUDE,DEW_ave,TEMP_ave,RH_ave,AT_ave,DEW_max,TEMP_max,RH_max,AT_max,sp_avg,tcc_avg,tp_sum,wind_speed_avg,wind_direction_deg_avg
time,,,,,,,,,,,,,,,,
1990-01-01,CA MAU,9.183333,105.15,21.76,24.46,85.71,28.91,22.2,28.3,92.95,32.42,101.036877,0.950259,8.196464e-04,2.603408,104.352150
1990-01-02,CA MAU,9.183333,105.15,21.76,24.46,85.71,28.91,22.2,28.3,92.95,32.42,101.126695,0.717658,1.756653e-05,3.751254,95.887684
1990-01-03,CA MAU,9.183333,105.15,21.53,23.78,87.31,28.07,21.7,24.9,91.82,29.30,101.225232,0.105999,2.860647e-07,4.373832,91.720210
1990-01-04,CA MAU,9.183333,105.15,20.90,23.68,84.61,27.57,21.5,25.3,91.27,29.57,101.261555,0.692104,3.038970e-04,4.438863,83.439938
1990-01-05,CA MAU,9.183333,105.15,22.16,25.70,81.55,30.41,23.0,29.0,94.68,33.25,101.322520,0.721015,0.000000e+00,2.595582,83.001394


In [8]:
df.tail()

,NAME,LATITUDE,LONGITUDE,DEW_ave,TEMP_ave,RH_ave,AT_ave,DEW_max,TEMP_max,RH_max,AT_max,sp_avg,tcc_avg,tp_sum,wind_speed_avg,wind_direction_deg_avg
time,,,,,,,,,,,,,,,,
2024-12-28,CA MAU,9.183333,105.15,24.2,27.01,85.08,33.19,26.2,29.6,95.95,35.58,101.284169,0.947528,0.001192,4.001243,69.316728
2024-12-29,CA MAU,9.183333,105.15,24.2,27.01,85.08,33.19,26.2,29.6,95.95,35.58,101.227320,0.597557,0.000457,3.198287,47.556545
2024-12-30,CA MAU,9.183333,105.15,24.2,27.01,85.08,33.19,26.2,29.6,95.95,35.58,101.234365,0.717204,0.000350,1.400367,153.289810
2024-12-31,CA MAU,9.183333,105.15,24.2,27.01,85.08,33.19,26.2,29.6,95.95,35.58,101.134755,0.988316,0.000016,1.832019,323.971547
2025-01-01,CA MAU,9.183333,105.15,24.2,27.01,85.08,33.19,26.2,29.6,95.95,35.58,101.012573,0.997857,0.000002,2.361484,347.759965


#### <span style="color:green">3.3. Data summary</span>

In [9]:
df.shape

(12785, 16)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 12785 entries, 1990-01-01 to 2025-01-01
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   NAME                    12785 non-null  object 
 1   LATITUDE                12785 non-null  float64
 2   LONGITUDE               12785 non-null  float64
 3   DEW_ave                 12785 non-null  float64
 4   TEMP_ave                12785 non-null  float64
 5   RH_ave                  12785 non-null  float64
 6   AT_ave                  12785 non-null  float64
 7   DEW_max                 12785 non-null  float64
 8   TEMP_max                12785 non-null  float64
 9   RH_max                  12785 non-null  float64
 10  AT_max                  12785 non-null  float64
 11  sp_avg                  12785 non-null  float64
 12  tcc_avg                 12785 non-null  float64
 13  tp_sum                  12785 non-null  float64
 14  wind_speed_avg       

In [11]:
df.describe()

,LATITUDE,LONGITUDE,DEW_ave,TEMP_ave,RH_ave,AT_ave,DEW_max,TEMP_max,RH_max,AT_max,sp_avg,tcc_avg,tp_sum,wind_speed_avg,wind_direction_deg_avg
count,12785.000000,1.278500e+04,12785.000000,12785.000000,12785.000000,12785.000000,12785.000000,12785.000000,12785.000000,12785.000000,12785.000000,12785.000000,12785.000000,12785.000000,12785.000000
mean,9.183333,1.051500e+02,23.832189,27.685349,80.653944,33.598047,24.832100,30.850196,92.509449,36.443160,100.975051,0.750865,0.005014,2.841517,163.685001
std,0.000000,1.421141e-14,1.464804,1.446314,6.460672,2.156398,1.449421,1.921500,4.462349,2.316119,0.185254,0.239361,0.005480,1.115032,77.805711
min,9.183333,1.051500e+02,14.410000,20.640000,55.190000,21.320000,15.700000,21.400000,57.670000,23.790000,100.355921,0.011411,0.000000,0.383929,5.304013
25%,9.183333,1.051500e+02,23.090000,26.730000,76.020000,32.250000,24.000000,29.700000,90.220000,35.010000,100.846086,0.595912,0.000522,1.990895,89.729649
50%,9.183333,1.051500e+02,24.250000,27.710000,80.410000,33.780000,25.200000,31.000000,93.560000,36.680000,100.956982,0.828511,0.003668,2.741878,148.766477
75%,9.183333,1.051500e+02,24.860000,28.680000,85.200000,35.130000,25.800000,32.200000,95.920000,38.070000,101.090514,0.958601,0.007686,3.578327,243.134185
max,9.183333,1.051500e+02,27.080000,32.830000,98.800000,40.040000,32.400000,44.000000,100.000000,49.530000,101.688842,1.000000,0.047215,7.083353,347.759965


### <span style="color:orange">5.Model</span>

#### <span style="color:green">5.1. Loading the dataset</span>

In [12]:
def create_sequences(data, seq_len=30, target_col=0):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])              # (seq_len, num_features)
        y.append(data[i+seq_len, target_col])    # chỉ 1 giá trị (univariate target)
    return np.array(X), np.array(y)

In [13]:
numeric_cols = df.select_dtypes(include='number').columns
df=df[numeric_cols][:1000]

scaler = StandardScaler()
features_scaled = scaler.fit_transform(df.values)

SEQ_LEN = 30
features, targets = create_sequences(features_scaled, SEQ_LEN, target_col=df.columns.get_loc("TEMP_max"))

features = features.reshape(features.shape[0], -1)
targets = targets.reshape(-1, 1) 

features = torch.tensor(features, dtype=torch.float32)
targets = torch.tensor(targets, dtype=torch.float32)

#### <span style="color:green">5.2. Splitting the dataset</span>

In [14]:
split_idx = int(len(df) * 0.8)
X_train, X_test = features[:split_idx], features[split_idx:]
y_train, y_test = targets[:split_idx], targets[split_idx:]

train_ds = TensorDataset(X_train, y_train)
test_ds  = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=False) 
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)

#### <span style="color:green">5.3. Building Models</span>

In [ ]:
def train_model_with_tb(units=64, 
                        dropout=0.2, 
                        lr=1e-3, 
                        batch_size=32, 
                        epochs=50, 
                        run_name="run"):

    step = 0
    model = nn.Sequential(
        nn.Linear(X_train.shape[1],units),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(units, 1)
    )
    model.train()
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    writer = SummaryWriter(log_dir=f"logs/{run_name}")

    writer.add_graph(model, next(iter(train_loader))[0])

    train_losses = []
    val_losses = []

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for data, targets in train_loader:
            optimizer.zero_grad()
            outputs = model(data)
            loss = criterion(outputs, targets)
            loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            train_loss += loss.item()
        
        avg_train_loss = train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        model.eval()
        val_loss = 0
        val_predictions = []
        val_targets = []
        
        with torch.no_grad():
            for data, targets in test_loader:
                outputs = model(data)
                loss = criterion(outputs, targets)
                val_loss += loss.item()
                
                val_predictions.extend(outputs.cpu().numpy())
                val_targets.extend(targets.cpu().numpy())
        
        avg_val_loss = val_loss / len(test_loader)
        val_losses.append(avg_val_loss)
        # Calculate additional metrics
        val_predictions = np.array(val_predictions)
        val_targets = np.array(val_targets)
        val_mae = mean_absolute_error(val_targets, val_predictions)
        val_rmse = np.sqrt(mean_squared_error(val_targets, val_predictions))
        
        # Log metrics to TensorBoard
        writer.add_scalar('Loss/Train', avg_train_loss, epoch)
        writer.add_scalar('Loss/Validation', avg_val_loss, epoch)
        writer.add_scalar('Metrics/MAE', val_mae, epoch)
        writer.add_scalar('Metrics/RMSE', val_rmse, epoch)
        writer.add_scalar('Learning_Rate', optimizer.param_groups[0]['lr'], epoch)
        
        # Log histograms of model parameters
        if epoch % 10 == 0:
            for name, param in model.named_parameters():
                writer.add_histogram(name, param, epoch)
                if param.grad is not None:
                    writer.add_histogram(f'{name}/grad', param.grad, epoch)


    writer.add_hparams(
        {
            "units": units,
            "dropout": dropout,
            "lr": lr,
            "batch_size": batch_size
        },
        {
            "mae": val_mae
        }
    )

    writer.close()
    return model

units = [32, 64]
dropouts = [0.0, 0.2]
lrs = [1e-3, 1e-4]
logdir = "logs/" + datetime.now().strftime("%Y%m%d-%H%M%S")
for un in units:
    for do in dropouts:
        for lr in lrs:
            run_name = f"{logdir}/hd{un}_do{do}_lr{lr}"
            print("Running:", run_name)
            train_model_with_tb(
                units=un, 
                dropout=do, 
                lr=lr,
                batch_size=32, 
                epochs=50, 
                run_name=run_name
                )


In [18]:
# logdir = "logs/" + datetime.now().strftime("%Y%m%d-%H%M%S")

# HP_NUM_UNITS = hp.HParam("num units", hp.Discrete([32, 64]))
# HP_DROPOUT = hp.HParam("dropout", hp.Discrete([0.1, 0.2]))
# HP_LR = hp.HParam("learning_rate", hp.Discrete([1e-3, 1e-4]))

# METRIC_MSE = 'mse'

# with tf.summary.create_file_writer(logdir).as_default():
#     hp.hparams_config(
#         hparams=[HP_NUM_UNITS, HP_DROPOUT, HP_LR],
#         metrics=[hp.Metric(METRIC_MSE, display_name='MSE')]
#     )

# def train_test_model(hparams, logdir):
#     units = hparams[HP_NUM_UNITS]
#     dropout = hparams[HP_DROPOUT]
#     lr = hparams[HP_LR]
    
#     model = Sequential([
#         Input(shape=(14,)),
#         Dense(units, activation='relu'),
#         Dropout(dropout),
#         Dense(1)
#     ])

#     model.compile(
#         optimizer=Adam(learning_rate=lr),
#         loss='mse',        
#         metrics=['mse']
#     )
    
#     model.fit(
#         X_train, 
#         y_train, 
#         epochs=50,
#         verbose=0,
#         # callbacks=[
#         #     keras.callbacks.TensorBoard(log_dir=logdir+"/scalars",
#         #                                 histogram_freq = 1,
#         #                                 profile_batch = '500,520'),
#         #     hp.KerasCallback(logdir, hparams),
#         # ]
#     )
    
#     loss, mse = model.evaluate(X_test, y_test, verbose=0)
#     print(f"Units: {units}, Dropout: {dropout}, Learning_rate: {lr} => MSE: {mse:.4f}")
#     run_dir = f"{logdir}/hparam_tuning/units_{units}, dropout_{dropout}, learning_rate_{lr}"
#     with tf.summary.create_file_writer(run_dir).as_default():
#         hp.hparams(hparams)
#         tf.summary.scalar(METRIC_MSE, mse, step=1)
#     return mse

# for lr in HP_LR.domain.values:
#     for units in HP_NUM_UNITS.domain.values:
#         for rate in HP_DROPOUT.domain.values:
#             hparams = {
#                 HP_LR: lr,
#                 HP_NUM_UNITS: units,
#                 HP_DROPOUT: rate,
#             }

#             train_test_model(hparams, logdir)

#### <span style="color:green">5.4. Training Models</span>

### <span style="color:orange">6.Model Evaluation</span>

In [19]:
# metrics = ['mae', 'mse', 'msle', 'mape']
# for metric in metrics:
#     plt.figure()
#     plt.plot(history.history[metric], label=f'Training {metric}')
#     plt.plot(history.history[f'val_{metric}'], label=f'Validation {metric}')
#     plt.title(metric.upper())
#     plt.xlabel('Epochs')
#     plt.ylabel(metric.upper())
#     plt.legend()
#     plt.show()